# Counterfactual Spatiotemporal Diffusion for Basketball — Full Run

End-to-end research run on Colab: **real SportVU ingestion → conditional diffusion training (GPU) → counterfactual stress-test → evaluation report**.

**Before running:** `Runtime` → `Change runtime type` → **T4 GPU**.

Then `Run all`. Details and troubleshooting: see `COLAB.md` in the repo.

In [ ]:
#@title 0. Configuration { display-mode: "form" }
REPO_URL = "https://github.com/your-org/basketball_diffusion.git"  #@param {type:"string"}
N_GAMES = 8  #@param {type:"integer"}
SYNTHETIC_SUPPLEMENT = 1500  #@param {type:"integer"}

GAMES = [
    "01.01.2016.CHA.at.TOR",
    "01.01.2016.DAL.at.MIA",
    "01.01.2016.NYK.at.CHI",
    "01.01.2016.ORL.at.WAS",
    "01.04.2016.IND.at.MIA",
    "01.04.2016.PHO.at.SAC",
    "01.06.2016.BOS.at.BKN",
    "01.06.2016.MIL.at.DAL",
    "01.06.2016.PHO.at.UTAH",
    "01.07.2016.ATL.at.CHI",
    "01.07.2016.BOS.at.WAS",
    "01.11.2016.MEM.at.HOU",
    "01.11.2016.MIL.at.SAS",
    "01.13.2016.CLE.at.LAC",
    "01.13.2016.DET.at.SAS",
    "01.15.2016.LAL.at.UTAH",
    "01.16.2016.DEN.at.MIA",
    "01.18.2016.MIA.at.MIL",
    "01.20.2016.LAC.at.CLE",
    "01.22.2016.CHI.at.BOS",
][:N_GAMES]
DATA_URL = "https://raw.githubusercontent.com/sealneaward/nba-movement-data/master/data/{}.7z"
print(f"{len(GAMES)} games queued; synthetic supplement {SYNTHETIC_SUPPLEMENT}")

In [ ]:
#@title 1. Get the code { display-mode: "form" }
import os, sys
if not os.path.exists('basketball_diffusion'):
    if 'your-org' in REPO_URL:
        raise RuntimeError(
            'Set REPO_URL to your fork, or upload the project zip and unzip it '
            'into basketball_diffusion/ (see COLAB.md).'
        )
    !git clone {REPO_URL}
os.chdir('basketball_diffusion')
sys.path.insert(0, os.getcwd())
print('workdir:', os.getcwd())
!ls

In [ ]:
#@title 2. Dependencies (torch is preinstalled on Colab — do NOT install it) { display-mode: "form" }
!pip install -q -r requirements-colab.txt
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', DEVICE)
if DEVICE == 'cpu':
    print('WARNING: no GPU attached — training will be slow. '
          'Runtime > Change runtime type > T4 GPU.')

In [ ]:
#@title 3. Download public SportVU games (7z archives) { display-mode: "form" }
import urllib.request, pathlib
pathlib.Path('data/raw').mkdir(parents=True, exist_ok=True)
for g in GAMES:
    dest = f'data/raw/{g}.7z'
    if os.path.exists(dest):
        print('have', dest)
        continue
    print('downloading', g)
    urllib.request.urlretrieve(DATA_URL.format(g), dest)
!ls -la data/raw | head -5

In [ ]:
#@title 4. Real-data ingestion pipeline { display-mode: "form" }
# Extracts .7z -> parses real moment schema -> meters -> team inference ->
# rim-centric frame -> Savitzky-Golay denoise -> screen detection ->
# drop/switch/blitz labels -> (N, 100, 11, 4) tensor cache.
!python scripts/pipeline.py --output-dir outputs --raw-dir data/raw --synthetic 0 2>&1 | tee ingest_log.txt

In [ ]:
#@title 5. Synthetic supplement (label balance) { display-mode: "form" }
# The strict geometric screen detector yields ~10-20 windows/game, and real
# blitz/switch counts are small. We mix in the behavioral simulator's labeled
# possessions so the conditional model sees all three schemes; real windows
# anchor realism. Set SYNTHETIC_SUPPLEMENT=0 to train real-only.
import numpy as np
from src.dataset import load_cache, save_cache, save_norm_stats
from src.synthetic_pnr import build_dataset

real = load_cache('outputs/processed_tensors')
n_real = len(real['traj'])
print('real windows:', n_real)
if SYNTHETIC_SUPPLEMENT > 0:
    syn = build_dataset(SYNTHETIC_SUPPLEMENT, seed=1)
    merged = {k: np.concatenate([syn[k], real[k]], axis=0) for k in ('traj', 'scheme', 'star')}
    save_cache('outputs/processed_tensors', merged)
    save_norm_stats('outputs/processed_tensors', merged['traj'])
    print('merged cache:', {k: v.shape for k, v in merged.items()})
    print('scheme counts:', np.bincount(merged['scheme'], minlength=3), '(drop/switch/blitz)')
    print('real fraction: %.2f' % (n_real / len(merged['traj'])))

In [ ]:
#@title 6. Train the conditional diffusion model { display-mode: "form" }
EPOCHS = 60  #@param {type:"integer"}
!python scripts/train.py --config configs/base.yaml --epochs {EPOCHS} --device {DEVICE} 2>&1 | tee train_log.txt

In [ ]:
#@title 7. Counterfactual stress-test + court animations { display-mode: "form" }
!python scripts/generate.py --ckpt outputs/checkpoints/last.pt --data-dir outputs/processed_tensors --out outputs/counterfactuals --n-per-scheme 16 --render --device {DEVICE}

In [ ]:
#@title 8. Evaluation report (FTD / ADE / compliance / vulnerability) { display-mode: "form" }
!python scripts/evaluate.py --ckpt outputs/checkpoints/last.pt --data-dir outputs/processed_tensors --n-samples 128 --device {DEVICE}
import json
print(json.dumps(json.load(open('outputs/eval/report.json')), indent=2))

In [ ]:
#@title 9. Watch generated possessions { display-mode: "form" }
import glob
from IPython.display import Video, display
clips = sorted(glob.glob('outputs/counterfactuals/*.mp4')) + sorted(glob.glob('outputs_real/*.mp4'))
for c in clips[:3]:
    print(c)
    display(Video(c, embed=True, width=480))

In [ ]:
#@title 10. Unit tests (sanity) { display-mode: "form" }
!python -m pytest tests/ -q

In [ ]:
#@title 10.5 Package artifacts for download to your machine { display-mode: "form" }
# Everything the paper needs: eval report, counterfactual arrays + MP4s,
# normalization stats, and the captured pipeline/training logs.
# Checkpoints (*.pt) are excluded — too heavy to be useful here.
!zip -q -r colab_artifacts.zip outputs/eval outputs/counterfactuals outputs/processed_tensors/norm_stats.npz ingest_log.txt train_log.txt -x "*.pt"
from google.colab import files
files.download('colab_artifacts.zip')
print('Drop colab_artifacts.zip into the local project folder and tell your agent to update the paper.')

In [ ]:
#@title 11. (Optional) Save artifacts to Google Drive { display-mode: "form" }
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/basketball_diffusion
    !cp -r outputs /content/drive/MyDrive/basketball_diffusion/
    print('saved to Drive: basketball_diffusion/outputs')

## Next steps

- **More games** → more real windows (raise `N_GAMES`; see the archive listing in `COLAB.md`).
- **Longer training** → better FTD/ADE; the checkpoint saves every epoch, so rerun cell 6 to continue.
- **Real-only training** → set `SYNTHETIC_SUPPLEMENT=0` once you have several hundred real windows.
- **Paper figures** → counterfactual MP4s (cell 7/9) are the qualitative figures; `report.json` (cell 8) feeds the results table.